In [1]:
"""
Generate 16-band slim feature dataset from raw data.
Pipeline: AEF deltas (04) → S1 before/after (04) → save full features → 
          extract slim 13 bands (05) → add 3 S1 temporal bands (06) → cleanup.

Optimized for multi-core CPU (Intel Xeon Platinum 8568Y+).
"""
from pathlib import Path
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
import re, os, warnings, gc
import geopandas as gpd
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing

warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

# ── CPU optimization ──
# ThreadPoolExecutor: numpy releases GIL so we get real parallelism; avoids pickle issues in notebooks
NUM_WORKERS = min(multiprocessing.cpu_count() - 2, 8)  # cap at 8 to avoid OOM
print(f"🖥️  Using {NUM_WORKERS} parallel workers (of {multiprocessing.cpu_count()} cores)")

# ── Paths (server layout) ──
_ROOT = Path("/root/makeathon-challenge-2026")
AEF_BASE  = _ROOT / "data/makeathon-challenge/aef-embeddings"
S1_BASE   = _ROOT / "data/makeathon-challenge/sentinel-1"
LABEL_DIR = _ROOT / "data/makeathon-challenge/labels/train"
META_DIR  = _ROOT / "data/makeathon-challenge/metadata"
OUT_DIR   = _ROOT / "data/cleaned/features"
SLIM_DIR  = _ROOT / "data/cleaned/features_slim"

BASELINE_YEAR = 2020
DELTA_YEARS = [2021, 2022, 2023, 2024, 2025]

KEEP_BANDS = [
    "aef_cosine_dist_2021", "aef_cosine_dist_2022", "aef_cosine_dist_2023",
    "aef_cosine_dist_2024", "aef_cosine_dist_2025",
    "aef_delta_norm_2021", "aef_delta_norm_2022", "aef_delta_norm_2023",
    "aef_delta_norm_2024", "aef_delta_norm_2025",
    "s1_change_mean", "s1_change_ratio", "s1_after_std",
]

NEW_BANDS = ["s1_max_drop", "s1_trend_slope", "s1_breakpoint_mag"]

# ── Tile lists ──
train_tiles = gpd.read_file(META_DIR / "train_tiles.geojson")["name"].tolist()
test_tiles  = gpd.read_file(META_DIR / "test_tiles.geojson")["name"].tolist()
print(f"Train: {len(train_tiles)} tiles, Test: {len(test_tiles)} tiles")

🖥️  Using 8 parallel workers (of 20 cores)
Train: 16 tiles, Test: 5 tiles


In [2]:
# ═══════════════════════════════════════════════════════════════
# STEP 1: AEF Delta Features (from notebook 04)
# ═══════════════════════════════════════════════════════════════

def compute_aef_features(tile_id, split="train"):
    """Compute AEF delta features: delta vectors, L2 norms, cosine distances vs 2020 baseline."""
    aef_dir = AEF_BASE / split
    base_path = aef_dir / f"{tile_id}_{BASELINE_YEAR}.tiff"

    with rasterio.open(base_path) as src:
        baseline = src.read().astype(np.float32)  # (64, H, W)
        profile = src.profile.copy()

    base_norm = np.linalg.norm(baseline, axis=0)
    base_norm = np.where(base_norm > 0, base_norm, np.nan)

    deltas, delta_norms, cosine_dists = {}, {}, {}
    for year in DELTA_YEARS:
        year_path = aef_dir / f"{tile_id}_{year}.tiff"
        if not year_path.exists():
            print(f"    ⚠ {year_path.name} missing, skipping")
            continue
        with rasterio.open(year_path) as src:
            emb = src.read().astype(np.float32)

        delta = emb - baseline
        deltas[year] = delta
        delta_norms[year] = np.linalg.norm(delta, axis=0)

        yr_norm = np.linalg.norm(emb, axis=0)
        yr_norm = np.where(yr_norm > 0, yr_norm, np.nan)
        dot = np.sum(baseline * emb, axis=0)
        cosine_dists[year] = 1.0 - dot / (base_norm * yr_norm)
        del emb

    return {'baseline': baseline, 'deltas': deltas, 'delta_norms': delta_norms,
            'cosine_dists': cosine_dists, 'profile': profile}

print("✅ compute_aef_features defined")

✅ compute_aef_features defined


In [3]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: S1 Before/After Features (from notebook 04)
# ═══════════════════════════════════════════════════════════════

def compute_s1_features(tile_id, split="train", before_years=(2020, 2021), after_years=(2023, 2024, 2025)):
    """Compute S1 temporal change features: before/after means, stds, change."""
    s1_dir = S1_BASE / split / f"{tile_id}__s1_rtc"
    if not s1_dir.exists():
        print(f"    ⚠ No S1 dir for {tile_id}")
        return None

    def get_period_files(years):
        files = []
        for f in sorted(s1_dir.glob("*.tif")):
            m = re.match(r".+__s1_rtc_(\d{4})_(\d{1,2})_(ascending|descending)\.tif", f.name)
            if m and int(m.group(1)) in years:
                files.append((f, int(m.group(1)), int(m.group(2)), m.group(3)))
        by_month = {}
        for f, yr, mo, orb in files:
            key = (yr, mo)
            if key not in by_month or orb == "descending":
                by_month[key] = f
        return list(by_month.values())

    before_files = get_period_files(before_years)
    after_files  = get_period_files(after_years)
    if not before_files or not after_files:
        print(f"    ⚠ Insufficient S1 files for {tile_id}")
        return None

    # Use first before-file as the single reference grid for ALL files
    with rasterio.open(before_files[0]) as src:
        ref_prof = src.profile.copy()
    ref_H, ref_W = ref_prof['height'], ref_prof['width']

    def read_stack_db(file_list):
        """Read S1 files into a dB stack, resampling any size mismatches to ref grid."""
        stack = []
        for f in file_list:
            with rasterio.open(f) as src:
                data = src.read(1).astype(np.float32)
            if data.shape == (ref_H, ref_W):
                stack.append(np.where(data > 0, 10 * np.log10(data), np.nan))
            else:
                resampled = np.full((ref_H, ref_W), np.nan, dtype=np.float32)
                with rasterio.open(f) as src:
                    reproject(source=src.read(1).astype(np.float32),
                              destination=resampled,
                              src_transform=src.transform, src_crs=src.crs,
                              dst_transform=ref_prof['transform'], dst_crs=ref_prof['crs'],
                              resampling=Resampling.bilinear)
                stack.append(np.where(resampled > 0, 10 * np.log10(resampled), np.nan))
        return np.array(stack)

    before_stack = read_stack_db(before_files)
    after_stack  = read_stack_db(after_files)
    s1_profile = ref_prof

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        before_mean = np.nanmean(before_stack, axis=0)
        before_std  = np.nanstd(before_stack, axis=0)
        after_mean  = np.nanmean(after_stack, axis=0)
        after_std   = np.nanstd(after_stack, axis=0)

    change_mean = after_mean - before_mean
    safe_before = np.where(np.abs(before_mean) > 0.01, before_mean, np.nan)
    change_ratio = after_mean / safe_before

    return {'before_mean': before_mean, 'before_std': before_std,
            'after_mean': after_mean, 'after_std': after_std,
            'change_mean': change_mean, 'change_ratio': change_ratio,
            's1_profile': s1_profile}

print("✅ compute_s1_features defined")

✅ compute_s1_features defined


In [4]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: Save Full Feature GeoTIFF per Tile (from notebook 04)
# ═══════════════════════════════════════════════════════════════

def save_tile_features(tile_id, split="train"):
    """Compute AEF + S1 features and save as multi-band GeoTIFF."""
    print(f"  [{tile_id}] AEF delta features...")
    aef = compute_aef_features(tile_id, split)
    H, W = aef['baseline'].shape[1], aef['baseline'].shape[2]
    aef_profile = aef['profile']

    print(f"  [{tile_id}] S1 before/after features...")
    s1 = compute_s1_features(tile_id, split)

    s1_bands_on_aef = {}
    s1_keys = ['before_mean', 'before_std', 'after_mean', 'after_std', 'change_mean', 'change_ratio']
    if s1 is not None:
        s1_h = s1['s1_profile']['height']
        s1_w = s1['s1_profile']['width']
        for key in s1_keys:
            src_arr = s1[key].reshape(1, s1_h, s1_w)
            dst_arr = np.full((1, H, W), np.nan, dtype=np.float32)
            reproject(source=src_arr, destination=dst_arr,
                      src_transform=s1['s1_profile']['transform'], src_crs=s1['s1_profile']['crs'],
                      dst_transform=aef_profile['transform'], dst_crs=aef_profile['crs'],
                      resampling=Resampling.bilinear)
            s1_bands_on_aef[key] = dst_arr[0]
    else:
        for key in s1_keys:
            s1_bands_on_aef[key] = np.full((H, W), np.nan, dtype=np.float32)

    # Stack all bands
    bands, band_names = [], []
    for i in range(64):
        bands.append(aef['baseline'][i]); band_names.append(f"aef_2020_b{i:02d}")
    for year in DELTA_YEARS:
        if year in aef['deltas']:
            for i in range(64):
                bands.append(aef['deltas'][year][i]); band_names.append(f"aef_delta_{year}_b{i:02d}")
        else:
            for i in range(64):
                bands.append(np.full((H, W), np.nan, dtype=np.float32)); band_names.append(f"aef_delta_{year}_b{i:02d}")
    for year in DELTA_YEARS:
        bands.append(aef['delta_norms'].get(year, np.full((H, W), np.nan, dtype=np.float32)))
        band_names.append(f"aef_delta_norm_{year}")
    for year in DELTA_YEARS:
        bands.append(aef['cosine_dists'].get(year, np.full((H, W), np.nan, dtype=np.float32)))
        band_names.append(f"aef_cosine_dist_{year}")
    for key in s1_keys:
        bands.append(s1_bands_on_aef[key]); band_names.append(f"s1_{key}")

    stack = np.array(bands, dtype=np.float32)

    out_path = OUT_DIR / split / f"{tile_id}_features.tif"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_profile = aef_profile.copy()
    out_profile.update(count=stack.shape[0], dtype='float32', compress='deflate',
                       nodata=np.nan, predictor=3, driver='GTiff')
    for k in ['blockxsize', 'blockysize', 'tiled']:
        out_profile.pop(k, None)

    with rasterio.open(out_path, 'w', **out_profile) as dst:
        dst.write(stack)
        dst.descriptions = tuple(band_names)

    size_mb = out_path.stat().st_size / 1024 / 1024
    print(f"  [{tile_id}] Saved {out_path.name} ({stack.shape[0]} bands, {size_mb:.1f} MB)")
    del aef, s1, stack, bands, s1_bands_on_aef
    gc.collect()
    return out_path, band_names

print("✅ save_tile_features defined")

✅ save_tile_features defined


In [5]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: Extract 13 Slim Bands (from notebook 05)
# ═══════════════════════════════════════════════════════════════

def extract_slim_features(tile_id, split):
    """Extract the 13 most useful bands from the full feature file."""
    src_path = OUT_DIR / split / f"{tile_id}_features.tif"
    dst_dir  = SLIM_DIR / split
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_path = dst_dir / f"{tile_id}_slim.tif"

    with rasterio.open(src_path) as src:
        descs = list(src.descriptions)
        name_to_idx = {d: i + 1 for i, d in enumerate(descs) if d}

        indices, found = [], []
        for b in KEEP_BANDS:
            if b in name_to_idx:
                indices.append(name_to_idx[b])
                found.append(b)
            else:
                print(f"    ⚠ {b} not found in {tile_id}")

        if not indices:
            raise ValueError(f"No matching bands found for {tile_id}! Available: {descs[:10]}...")

        data = src.read(indices)
        profile = src.profile.copy()
        profile.update(count=len(indices), compress="deflate", predictor=3)

    with rasterio.open(dst_path, "w", **profile) as dst:
        dst.write(data)
        dst.descriptions = tuple(found)

    mb = dst_path.stat().st_size / 1e6
    print(f"  [{tile_id}] Slim: {len(found)} bands, {mb:.1f} MB")

print("✅ extract_slim_features defined")

✅ extract_slim_features defined


In [6]:
# ═══════════════════════════════════════════════════════════════
# STEP 5: S1 Temporal Features + Append to Slim (from notebook 06)
# ═══════════════════════════════════════════════════════════════

def load_s1_timeseries(tile_id, split="train"):
    """Load monthly S1 backscatter as a time-ordered dB stack. Prefers descending orbit."""
    s1_dir = S1_BASE / split / f"{tile_id}__s1_rtc"
    if not s1_dir.exists():
        return None, None, None

    files = []
    for f in sorted(s1_dir.glob("*.tif")):
        m = re.match(r".+__s1_rtc_(\d{4})_(\d{1,2})_(ascending|descending)\.tif", f.name)
        if m:
            files.append((f, int(m.group(1)), int(m.group(2)), m.group(3)))

    by_month = {}
    for f, yr, mo, orb in files:
        key = (yr, mo)
        if key not in by_month or orb == "descending":
            by_month[key] = f

    sorted_keys = sorted(by_month.keys())
    if not sorted_keys:
        return None, None, None

    with rasterio.open(by_month[sorted_keys[0]]) as src:
        H, W = src.height, src.width
        profile = {'transform': src.transform, 'crs': src.crs, 'height': H, 'width': W}

    stack = np.full((len(sorted_keys), H, W), np.nan, dtype=np.float32)
    for i, key in enumerate(sorted_keys):
        with rasterio.open(by_month[key]) as src:
            data = src.read(1).astype(np.float32)
            if data.shape == (H, W):
                stack[i] = np.where(data > 0, 10 * np.log10(data), np.nan)
    return stack, sorted_keys, profile


def compute_s1_temporal_features(stack, times):
    """Compute max_drop, trend_slope, breakpoint_mag from (T, H, W) dB stack.
    Fully vectorized – no per-pixel Python loops."""
    T, H, W = stack.shape
    flat = stack.reshape(T, -1)  # (T, N)

    # ── Max drop (vectorized) ──
    diffs = np.diff(flat, axis=0)
    max_drop = np.nanmin(diffs, axis=0)

    # ── Trend slope (vectorized) ──
    t_idx = np.arange(T, dtype=np.float32)
    t_mean = t_idx.mean()
    t_centered = t_idx - t_mean                     # (T,)
    t_var = np.sum(t_centered ** 2)                  # scalar

    x_mean = np.nanmean(flat, axis=0)               # (N,)
    x_centered = flat - x_mean[np.newaxis, :]       # (T, N)
    # mask NaN contributions
    valid_mask = np.isfinite(x_centered)
    x_centered = np.where(valid_mask, x_centered, 0.0)

    numerator = t_centered[:, np.newaxis] * x_centered  # (T, N)
    numerator = np.sum(numerator, axis=0)               # (N,)

    # denominator: sum of t_centered^2 only where valid
    denom = np.sum(t_centered[:, np.newaxis] ** 2 * valid_mask, axis=0)
    trend_slope = np.where(denom > 0, numerator / denom, 0.0).astype(np.float32)

    # ── Breakpoint magnitude (vectorized CUSUM) ──
    deviations = np.where(np.isfinite(flat - x_mean[np.newaxis, :]),
                          flat - x_mean[np.newaxis, :], 0.0)
    cusum = np.cumsum(deviations, axis=0)            # (T, N)
    bp_idx = np.argmax(np.abs(cusum), axis=0)        # (N,)

    # Vectorized before/after mean at breakpoint
    N = flat.shape[1]
    # Create masks for before/after breakpoint
    time_indices = np.arange(T)[:, np.newaxis]       # (T, 1)
    bp_row = bp_idx[np.newaxis, :]                   # (1, N)

    before_mask = (time_indices < bp_row) & np.isfinite(flat)
    after_mask  = (time_indices >= bp_row) & np.isfinite(flat)

    # Safe means with masking
    before_sum = np.where(before_mask, flat, 0.0).sum(axis=0)
    before_cnt = before_mask.sum(axis=0).astype(np.float32)
    after_sum  = np.where(after_mask, flat, 0.0).sum(axis=0)
    after_cnt  = after_mask.sum(axis=0).astype(np.float32)

    before_mean = np.where(before_cnt > 0, before_sum / before_cnt, 0.0)
    after_mean  = np.where(after_cnt > 0, after_sum / after_cnt, 0.0)
    breakpoint_mag = (after_mean - before_mean).astype(np.float32)

    # Zero out where breakpoint is too close to edges
    edge_mask = (bp_idx < 2) | (bp_idx > T - 3)
    breakpoint_mag[edge_mask] = 0.0

    return {'s1_max_drop': max_drop.reshape(H, W),
            's1_trend_slope': trend_slope.reshape(H, W),
            's1_breakpoint_mag': breakpoint_mag.reshape(H, W)}


def add_s1_temporal(tile_id, split):
    """Load S1 timeseries, compute temporal features, append to slim file."""
    stack, times, s1_prof = load_s1_timeseries(tile_id, split)
    if stack is None:
        print(f"  [5] ⚠ No S1 timeseries for {tile_id}, filling NaN")
        slim_path = SLIM_DIR / split / f"{tile_id}_slim.tif"
        with rasterio.open(slim_path) as src:
            old_data = src.read()
            old_descs = list(src.descriptions)
            profile = src.profile.copy()
            H, W = src.height, src.width
        new_bands = [np.full((H, W), np.nan, dtype=np.float32) for _ in NEW_BANDS]
        combined = np.concatenate([old_data, np.array(new_bands)], axis=0)
        profile.update(count=combined.shape[0], compress="deflate", predictor=3)
        with rasterio.open(slim_path, "w", **profile) as dst:
            dst.write(combined)
            dst.descriptions = tuple(old_descs + NEW_BANDS)
        return

    s1_feats = compute_s1_temporal_features(stack, times)

    slim_path = SLIM_DIR / split / f"{tile_id}_slim.tif"
    with rasterio.open(slim_path) as src:
        old_data = src.read()
        old_descs = list(src.descriptions)
        profile = src.profile.copy()
        dst_tf, dst_crs = src.transform, src.crs
        H, W = src.height, src.width

    s1_H, s1_W = s1_prof['height'], s1_prof['width']
    new_bands = []
    for name in NEW_BANDS:
        src_arr = s1_feats[name].reshape(1, s1_H, s1_W)
        dst_arr = np.full((1, H, W), np.nan, dtype=np.float32)
        reproject(source=src_arr, destination=dst_arr,
                  src_transform=s1_prof['transform'], src_crs=s1_prof['crs'],
                  dst_transform=dst_tf, dst_crs=dst_crs,
                  resampling=Resampling.bilinear)
        new_bands.append(dst_arr[0])

    combined = np.concatenate([old_data, np.array(new_bands)], axis=0)
    all_descs = old_descs + NEW_BANDS
    profile.update(count=combined.shape[0], compress="deflate", predictor=3)

    with rasterio.open(slim_path, "w", **profile) as dst:
        dst.write(combined)
        dst.descriptions = tuple(all_descs)

    mb = slim_path.stat().st_size / 1e6
    print(f"  [5] Temporal: {combined.shape[0]} bands total, {mb:.1f} MB")
    del stack, s1_feats, old_data, combined
    gc.collect()

print("✅ add_s1_temporal defined (vectorized)")

✅ add_s1_temporal defined (vectorized)


In [7]:
# ═══════════════════════════════════════════════════════════════
# STEP 6: Process All Tiles (PARALLELIZED)
# ═══════════════════════════════════════════════════════════════

def process_single_tile(args):
    """Process one tile end-to-end. Designed for multiprocessing."""
    tile, split = args
    import warnings
    warnings.filterwarnings("ignore")
    try:
        save_tile_features(tile, split)
        extract_slim_features(tile, split)
        add_s1_temporal(tile, split)

        # Cleanup large full-feature file
        full_path = OUT_DIR / split / f"{tile}_features.tif"
        if full_path.exists():
            full_path.unlink()

        return (tile, split, True, None)
    except Exception as e:
        import traceback
        return (tile, split, False, traceback.format_exc())


for split, tile_list in [("train", train_tiles), ("test", test_tiles)]:
    print(f"\n{'='*60}")
    print(f"  PROCESSING {split.upper()} ({len(tile_list)} tiles) — {NUM_WORKERS} workers")
    print(f"{'='*60}")

    tasks = [(tile, split) for tile in tile_list]
    done, failed = 0, 0

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {executor.submit(process_single_tile, t): t for t in tasks}
        for future in as_completed(futures):
            tile, sp, success, err = future.result()
            done += 1
            if success:
                print(f"  ✅ [{done}/{len(tasks)}] {tile} complete")
            else:
                failed += 1
                print(f"  ❌ [{done}/{len(tasks)}] {tile} FAILED:\n{err}")

    print(f"\n  Done: {done - failed} OK, {failed} failed")

# ── Summary ──
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for split in ["train", "test"]:
    d = SLIM_DIR / split
    if d.exists():
        files = list(d.glob("*_slim.tif"))
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f"  {split}: {len(files)} files, {total_mb:.0f} MB total")
        for f in files:
            with rasterio.open(f) as src:
                if src.count != 16:
                    print(f"    ⚠ {f.name}: {src.count} bands (expected 16)")


  PROCESSING TRAIN (16 tiles) — 8 workers
  [47QMB_0_8] AEF delta features...
  [48PWV_7_8] AEF delta features...
  [48PUT_0_8] AEF delta features...
  [48QWD_2_2] AEF delta features...
  [48PXC_7_7] AEF delta features...
  [48PYB_3_6] AEF delta features...
  [47QQV_2_4] AEF delta features...
  [48QVE_3_0] AEF delta features...


  [47QMB_0_8] S1 before/after features...
  [47QQV_2_4] S1 before/after features...
  [48PUT_0_8] S1 before/after features...
  [48PWV_7_8] S1 before/after features...
  [48QWD_2_2] S1 before/after features...
  [48PYB_3_6] S1 before/after features...
  [48QVE_3_0] S1 before/after features...
  [48PXC_7_7] S1 before/after features...
  [47QQV_2_4] Saved 47QQV_2_4_features.tif (400 bands, 1147.5 MB)
  [47QMB_0_8] Saved 47QMB_0_8_features.tif (400 bands, 1115.3 MB)
  [48PWV_7_8] Saved 48PWV_7_8_features.tif (400 bands, 1152.3 MB)
  [48QVE_3_0] Saved 48QVE_3_0_features.tif (400 bands, 1144.8 MB)
  [48QWD_2_2] Saved 48QWD_2_2_features.tif (400 bands, 1152.0 MB)
  [48PYB_3_6] Saved 48PYB_3_6_features.tif (400 bands, 1159.8 MB)
  [48PXC_7_7] Saved 48PXC_7_7_features.tif (400 bands, 1149.1 MB)
  [48PUT_0_8] Saved 48PUT_0_8_features.tif (400 bands, 1153.5 MB)
  [47QQV_2_4] Slim: 13 bands, 36.8 MB
  [48PWV_7_8] Slim: 13 bands, 37.1 MB
  [47QMB_0_8] Slim: 13 bands, 35.1 MB
  [48QVE_3_0] Slim: 13